In [68]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

In [69]:
df = pd.read_csv('raw_model.csv')
df.head()

,First_pokemon,Second_pokemon,Winner,ID_P1,Name_P1,Type 1_P1,Type 2_P1,Type_P1,Abilities_P1,HiddenAbility_P1,...,DamageFromSteel_P2,DamageFromFire_P2,DamageFromWater_P2,DamageFromGrass_P2,DamageFromElectric_P2,DamageFromPsychic_P2,DamageFromIce_P2,DamageFromDragon_P2,DamageFromDark_P2,DamageFromFairy_P2
0,266,298,298,266.0,Larvitar,Rock,Ground,"['Rock', 'Ground']",['Guts'],['Sand Veil'],...,1.0,2.0,0.5,0.5,0.5,0.0,2.0,1.0,0.5,2.0
1,702,701,701,702.0,Virizion,Grass,Fighting,"['Grass', 'Fighting']",['Justified'],[],...,2.0,0.5,2.0,2.0,1.0,2.0,1.0,1.0,0.5,2.0
2,191,668,668,191.0,Togetic,Fairy,Flying,"['Fairy', 'Flying']","['Hustle', 'Serene Grace']",['Super Luck'],...,1.0,1.0,1.0,1.0,1.0,0.5,1.0,1.0,2.0,1.0
3,237,683,683,237.0,Slugma,Fire,NaN,['Fire'],"['Magma Armor', 'Flame Body']",['Weak Armor'],...,1.0,0.5,0.5,0.5,0.5,1.0,2.0,2.0,1.0,2.0
4,151,231,151,151.0,Omastar,Rock,Water,"['Rock', 'Water']","['Swift Swim', 'Shell Armor']",['Weak Armor'],...,2.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [70]:
df.duplicated().sum()

1952

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
# Tạo nhãn: 0 nếu P1 thắng, 1 nếu P2 thắng (chỉ dùng thông tin của trận đó)
df['Winner'] = np.where(df['Winner'] == df['First_pokemon'], 0, 1)
obj_cols = df.select_dtypes(include='object').columns
df_num = df.drop(columns=obj_cols)

X = df_num.drop(columns=['First_pokemon', 'Second_pokemon', 'Winner'])
y = df_num['Winner']

# 2. Chia train/val/test trước, rồi mới preprocessing → tránh leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

print("Số features:", X.shape[1])
print("Train/Val/Test:", len(X_train), len(X_val), len(X_test))

# 3. Định nghĩa 3 mô hình (preprocessing nằm trong Pipeline → fit CHỈ trên train)

# Logistic Regression: impute + scale bên trong pipeline 
logreg_clf = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=100, n_jobs=-1))
])

# Random Forest: tree-based
rf_clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# XGBoost: hỗ trợ NaN native, không cần impute ngoài; fit chỉ trên train
xgb_clf = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)

models = {
    "Logistic Regression": logreg_clf,
    "Random Forest": rf_clf,
    "XGBoost": xgb_clf
}

# 4. Train & evaluate
for name, model in models.items():
    model.fit(X_train, y_train)

    print(f"\n{name}")
    for split_name, X_split, y_split in [
        ("Train", X_train, y_train),
        ("Val",   X_val,   y_val),
        ("Test",  X_test,  y_test),
    ]:
        y_pred = model.predict(X_split)
        acc = accuracy_score(y_split, y_pred)
        print(f"{split_name}: accuracy = {acc:.4f}")


Số features: 73
Train/Val/Test: 32672 5766 9610

Logistic Regression
Train: accuracy = 0.8419
Val: accuracy = 0.8521
Test: accuracy = 0.8548

Random Forest
Train: accuracy = 0.9789
Val: accuracy = 0.8848
Test: accuracy = 0.8850

XGBoost
Train: accuracy = 0.8856
Val: accuracy = 0.8920
Test: accuracy = 0.8938
